In [1]:
import pandas as pd
import json
import ast
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import seaborn as sns
import numpy as np
from scipy.optimize import curve_fit
from tqdm import tqdm
from tqdm.auto import tqdm
tqdm.pandas()
import os
import math
import psutil
import gc

df_moves = pd.read_parquet('df_moves5.parquet')

In [2]:
df_moves.columns

Index(['uid', 'Move_Idx', 'Player', 'Color', 'E', 'E1', 'E2', 'E3', 'E4', 'E5',
       'Judgment_Raw', 'Is_Error', 'Is_Blunder', 'Δi', 'Δ', 'ELO', 'ELO_Group',
       'Is_Forced', 'Is_Optimal', 'Remain_Time', 'Time_Group', 'Move_Time',
       'Game_Length', 'Progress', 'Relative_Stage(temp)', 'P_model', 'S'],
      dtype='object')

In [3]:
df_moves.head()

,uid,Move_Idx,Player,Color,E,E1,E2,E3,E4,E5,...,Is_Forced,Is_Optimal,Remain_Time,Time_Group,Move_Time,Game_Length,Progress,Relative_Stage(temp),P_model,S
0,34,1,Dennis Wagner,White,33.0,36.0,34,31,25,16,...,0,1,179.800003,Rich (>170),1.2,63,0.015873,Opening (0-33%),0.434,1.203
1,34,2,Petar Ratkovic,Black,36.0,33.0,34,34,42,48,...,0,1,177.800003,Rich (>170),3.2,63,0.031746,Opening (0-33%),0.380,1.396
2,34,3,Dennis Wagner,White,30.0,36.0,33,27,25,20,...,0,1,178.500000,Rich (>170),2.3,63,0.047619,Opening (0-33%),0.436,1.199
3,34,4,Petar Ratkovic,Black,44.0,30.0,63,66,78,79,...,0,0,178.699997,Rich (>170),0.1,63,0.063492,Opening (0-33%),0.422,1.243
4,34,5,Dennis Wagner,White,32.0,44.0,35,26,21,21,...,0,0,179.199997,Rich (>170),0.3,63,0.079365,Opening (0-33%),0.446,1.165


In [8]:
tolerance_cp = 10

df_moves['Is_Optimal'] = np.where(df_moves['Δi'] <= tolerance_cp, 1, 0).astype('int8')
df_moves['Game_Length'] = df_moves.groupby('uid')['Move_Idx'].transform('max').astype('int16')
df_moves['Progress'] = (df_moves['Move_Idx'] / df_moves['Game_Length']).astype('float32')


bins = [-np.inf, 0.33, 0.66, np.inf]
labels = ['Opening (0-33%)', 'Middlegame (33-66%)', 'Endgame (66-100%)']
df_moves['Relative_Stage(temp)'] = pd.cut(
    df_moves['Progress'], 
    bins=bins, 
    labels=labels, 
    right=True  # right=True 表示区间是 (a, b]，即包含 0.33 和 0.66
).astype('category')

# 检查内存占用情况
print(df_moves[['Is_Optimal', 'Game_Length', 'Progress', 'Relative_Stage(temp)']].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62435670 entries, 0 to 62435669
Data columns (total 4 columns):
 #   Column                Dtype   
---  ------                -----   
 0   Is_Optimal            int8    
 1   Game_Length           int16   
 2   Progress              float32 
 3   Relative_Stage(temp)  category
dtypes: category(1), float32(1), int16(1), int8(1)
memory usage: 476.3 MB
None


# 大数据集跑出的结果    

### Summary of Fitting Parameters

| ELO Group | A | τ | x₀ | A_err | τ_err | x₀_err | Max CI Width | d_max | min_cnt |
|-----------|------|------|------|------|------|------|------|------|------|
| ≥3000 | 0.884 | 166.9 | -5.8 | 0.0024 | 2.94 | 1.91 | 0.0109 | 700 | 50 |
| 2800–3000 | 0.879 | 173.1 | 10.0 | 0.0020 | 2.24 | 1.34 | 0.0085 | 700 | 50 |
| 2600–2800 | 0.872 | 172.7 | 29.6 | 0.0018 | 1.91 | 1.10 | 0.0069 | 700 | 80 |
| 2400–2600 | 0.861 | 171.6 | 44.4 | 0.0019 | 1.86 | 1.07 | 0.0069 | 700 | 100 |
| 2200–2400 | 0.849 | 171.5 | 56.0 | 0.0023 | 2.21 | 1.28 | 0.0085 | 700 | 100 |
| 2000–2200 | 0.837 | 169.2 | 66.3 | 0.0034 | 3.17 | 1.88 | 0.0129 | 700 | 50 |
| <2000 | 0.855 | 181.3 | 91.5 | 0.0113 | 6.71 | 4.90 | 0.0176 | 700 | 50 |

# P_model specific part
## if forced move ----- delta = NaN ----- S = NaN, P = NaN

In [10]:
elo_params = {
    '>=3000':      {'A': 0.884, 'τ': 166.9, 'x0': -5.8},
    '2800-3000':   {'A': 0.879, 'τ': 173.1, 'x0': 10.0},
    '2600-2800':   {'A': 0.872, 'τ': 172.7, 'x0': 29.6},
    '2400-2600':   {'A': 0.861, 'τ': 171.6, 'x0': 44.4},
    '2200-2400':   {'A': 0.849, 'τ': 171.5, 'x0': 56.0},
    '2000-2200':   {'A': 0.837, 'τ': 169.2, 'x0': 66.3},
    '<2000':       {'A': 0.855, 'τ': 181.3, 'x0': 91.5}
}
params_df = pd.DataFrame.from_dict(elo_params, orient='index').reset_index()
params_df = params_df.rename(columns={'index': 'ELO_Group'})
params_df['A'] = params_df['A'].astype('float32')
params_df['τ'] = params_df['τ'].astype('float32')
params_df['x0'] = params_df['x0'].astype('float32')

df_moves['ELO_Group'] = df_moves['ELO_Group'].astype('category')
if 'Δ' in df_moves.columns:
    df_moves['Δ'] = df_moves['Δ'].astype('float32')

df_moves = df_moves.merge(params_df, on='ELO_Group', how = 'left')
df_moves['P_model'] = (df_moves['A'] / (1 + np.exp(-(df_moves['Δ'] - df_moves['x0']) / df_moves['τ']))).astype('float32')
df_moves['S'] = (-np.log2(df_moves['P_model'])).astype('float32')
df_moves['x0'] = df_moves['x0'].round(1)
df_moves['P_model'] = df_moves['P_model'].round(3)
df_moves['S'] = df_moves['S'].round(3)

# 把A τ x0扔了
df_moves = df_moves.drop(columns=['A', 'τ', 'x0'])
df_moves.to_parquet('df_moves4.parquet', index=False, engine='pyarrow')

print(df_moves[['ELO_Group', 'Δ', 'P_model', 'S']].head())

   ELO_Group     Δ  P_model      S
0  2800-3000   6.0    0.434  1.203
1  2400-2600   4.0    0.380  1.396
2  2800-3000   7.0    0.436  1.199
3  2400-2600  38.0    0.422  1.243
4  2800-3000  15.0    0.446  1.165


In [12]:
df_moves.head()

,uid,Move_Idx,Player,Color,E,E1,E2,E3,E4,E5,...,Is_Forced,Is_Optimal,Remain_Time,Time_Group,Move_Time,Game_Length,Progress,Relative_Stage(temp),P_model,S
0,34,1,Dennis Wagner,White,33.0,36.0,34,31,25,16,...,0,1,179.800003,Rich (>170),1.2,63,0.015873,Opening (0-33%),0.434,1.203
1,34,2,Petar Ratkovic,Black,36.0,33.0,34,34,42,48,...,0,1,177.800003,Rich (>170),3.2,63,0.031746,Opening (0-33%),0.380,1.396
2,34,3,Dennis Wagner,White,30.0,36.0,33,27,25,20,...,0,1,178.500000,Rich (>170),2.3,63,0.047619,Opening (0-33%),0.436,1.199
3,34,4,Petar Ratkovic,Black,44.0,30.0,63,66,78,79,...,0,0,178.699997,Rich (>170),0.1,63,0.063492,Opening (0-33%),0.422,1.243
4,34,5,Dennis Wagner,White,32.0,44.0,35,26,21,21,...,0,0,179.199997,Rich (>170),0.3,63,0.079365,Opening (0-33%),0.446,1.165


In [2]:
# df_moves.to_parquet('df_moves4.parquet')
process = psutil.Process(os.getpid())
print(process.memory_info().rss / 1024**3, "GB")

15.10818099975586 GB


In [2]:
%whos

Variable             Type         Data/Info
-------------------------------------------
ast                  module       <module 'ast' from 'c:\\U<...>\Python312\\Lib\\ast.py'>
curve_fit            function     <function curve_fit at 0x0000021AC637D6C0>
df_moves             DataFrame                uid  Move_Idx<...>435670 rows x 27 columns]
gc                   module       <module 'gc' (built-in)>
json                 module       <module 'json' from 'c:\\<...>\Lib\\json\\__init__.py'>
make_interp_spline   function     <function make_interp_spl<...>ne at 0x0000021AC65A65C0>
math                 module       <module 'math' (built-in)>
np                   module       <module 'numpy' from 'c:\<...>ges\\numpy\\__init__.py'>
os                   module       <module 'os' (frozen)>
pd                   module       <module 'pandas' from 'c:<...>es\\pandas\\__init__.py'>
plt                  module       <module 'matplotlib.pyplo<...>\\matplotlib\\pyplot.py'>
psutil               module 

# P_model original part
## if forced move ----- delta = NaN ----- S = NaN, P = NaN

In [3]:
GLOBAL_τ_List = [10.0, 50.0, 100.0, 150.0]

# 将底层的 Δ 数组抽出来并转换为 float32
delta_values = df_moves['Δ'].values.astype(np.float32)
for τ_group in GLOBAL_τ_List:
    p_col_name = f'P_model(Global={τ_group}cp)'
    s_col_name = f'S(Global={τ_group}cp)'
    
    # 计算 P_model float32
    p_values = (1 / (1 + np.exp(-delta_values / τ_group))).astype(np.float32)
    # 计算 S
    # np.clip(p_values, 1e-9, 1.0) 会直接在底层将所有小于 1e-9 的值“托底”到 1e-9
    s_values = -np.log2(np.clip(p_values, 1e-9, 1.0)).astype(np.float32)
    # 赋值回df
    df_moves[p_col_name] = np.round(p_values, 2)
    df_moves[s_col_name] = np.round(s_values, 2)

print("新增列：")
print([col for col in df_moves.columns if "Global=" in col])

df_moves.to_parquet('df_moves5.parquet')

新增列：
['P_model(Global=10.0cp)', 'S(Global=10.0cp)', 'P_model(Global=50.0cp)', 'S(Global=50.0cp)', 'P_model(Global=100.0cp)', 'S(Global=100.0cp)', 'P_model(Global=150.0cp)', 'S(Global=150.0cp)']


In [4]:
df_moves['Move_Time'].head(10)

0    1.2
1    3.2
2    2.3
3    0.1
4    0.3
5    1.4
6    1.1
7    1.0
8    0.5
9    0.8
Name: Move_Time, dtype: float32

# cognitive speed part

$$
\text{Cognitive\_Speed}_{\text{each move}} = \frac{Δ_{\text{each move}}}{t_{\text{each move}}}
$$    
                                                             only calculate from 'Is_Optimal == 1' moves

耗时为0的步子，认知速度设为了inf    
画图的时候舍弃被设为inf的走子，同时只话Is_Optimal = 1的走子

In [3]:
df_moves['Move_Time'].describe()

count    6.243567e+07
mean     4.323132e+00
std      6.617140e+00
min      0.000000e+00
25%      9.000000e-01
50%      1.900000e+00
75%      4.900000e+00
max      1.821000e+02
Name: Move_Time, dtype: float64

In [4]:
(df_moves['Move_Time']==0).mean()

np.float64(1.6016485448142062e-08)

In [5]:
# 提取基础特征并保持 float32
move_time_vals = df_moves['Move_Time'].values.astype(np.float32)
delta_vals = df_moves['Δ'].values.astype(np.float32)
with np.errstate(divide='ignore', invalid='ignore'):
    cog_speed = delta_vals / move_time_vals

# 把所有耗时为 0 的步数，其认知速度强制设为正无穷大 (np.inf)
cog_speed = np.where(move_time_vals == 0, np.inf, cog_speed)

df_moves['Cog_Speed'] = np.round(cog_speed, 2).astype(np.float32)
df_moves.to_parquet('df_moves6.parquet')
print(df_moves[['Move_Time', 'Δ', 'Cog_Speed']].head(10))

   Move_Time     Δ  Cog_Speed
0        1.2   6.0       5.00
1        3.2   4.0       1.25
2        2.3   7.0       3.04
3        0.1  38.0     380.00
4        0.3  15.0      50.00
5        1.4  22.0      15.71
6        1.1   4.0       3.64
7        1.0  18.0      18.00
8        0.5  21.0      42.00
9        0.8  17.0      21.25


扔掉后面不用的列

In [7]:
df_moves.columns

Index(['uid', 'Move_Idx', 'Player', 'Color', 'E', 'E1', 'E2', 'E3', 'E4', 'E5',
       'Judgment_Raw', 'Is_Error', 'Is_Blunder', 'Δi', 'Δ', 'ELO', 'ELO_Group',
       'Is_Forced', 'Is_Optimal', 'Remain_Time', 'Time_Group', 'Move_Time',
       'Game_Length', 'Progress', 'Relative_Stage(temp)', 'P_model', 'S',
       'P_model(Global=10.0cp)', 'S(Global=10.0cp)', 'P_model(Global=50.0cp)',
       'S(Global=50.0cp)', 'P_model(Global=100.0cp)', 'S(Global=100.0cp)',
       'P_model(Global=150.0cp)', 'S(Global=150.0cp)', 'Cog_Speed'],
      dtype='object')

In [ ]:
df_moves = df_moves.drop(columns=['E1', 'E2', 'E3', 'E4', 'E5'])
# df_moves = df_moves.drop(columns=['E'])

另开一个notebook去画图？

实测数据在 Δ 接近 0 时准确率极高，随后迅速下降，这在国际象棋中被称为**“简单局面悖论”**：
 
高准确率起点 ( Δ 接近 0 )：当 Δ 非常小时，意味着局面处于“非关键期”。此时最好的两步棋评分几乎一样，根据你的定义（容差 $10cp$），玩家只要不走错得离谱，几乎随便走一步都是“最优”的 。此外，这也包含了大量的开局库（Book moves）和强制招法（Forced moves），这些棋步几乎不需要思考就能走对 。     

触底下降区：随着 Δ 略微增大，局面进入“关键期（Critical positions）”。此时正确的路径只有一条，而替代方案虽然只差一点点，但已经足以致命。人类在面对这种“微妙但重要”的差异时最容易犯错，因此准确率降至最低 。     

逐步回升区：当 Δ 变得很大（如 $200cp$ 以上）时，最优解变得非常“直观”（例如白捡一个子），此时准确率才开始遵循你的理论模型，随难度的降低而回升 。

原始不分箱子的，可能会有很多毛刺。    
换大数据集的时候同样记得改delta_col的范围为1000-1500cp

认知速度的具体画图与分析    
### distribution: very long tail

In [ ]:
speed_df = time_complex['Cognitive_Speed']
plt.figure(figsize=(5,3))
plt.hist(speed_df, bins=50, edgecolor='black', alpha=0.8)

plt.xlabel('Cognitive_Speed')
plt.ylabel('Count')
plt.title('Distribution of Cognitive_Speed (Is_Optimal = 1)')

plt.grid(alpha=0.3)
plt.show()

counts, edges = np.histogram(speed_df, bins = 50)
for i in range(len(counts)):
    print(f"Bin: {edges[i]:.2f}  -  {edges[i+1]:.2f}  Counts: {counts[i]}")

In [ ]:
speed_focus0 = speed_df[speed_df <= 200]
speed_focus1 = speed_df[speed_df <= 20]
# 0-200有85%的数据
plt.figure(figsize=(5,3))
plt.hist(speed_focus0, bins=100, edgecolor='black', alpha=0.8)
plt.xlabel('Cognitive_Speed')
plt.ylabel('Count')
plt.title('Distribution of Cognitive_Speed (0–500)')
plt.grid(alpha=0.3)
plt.show()
# 0-20有50%的数据
plt.figure(figsize=(5,3))
plt.hist(speed_focus1, bins=100, edgecolor='black', alpha=0.8)
plt.xlabel('Cognitive_Speed')
plt.ylabel('Count')
plt.title('Distribution of Cognitive_Speed (0–500)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(speed_df.describe())

In [ ]:
elo_order = ['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']
distinct_palette = sns.color_palette("Set1", len(elo_order))
color_map = dict(zip(elo_order, distinct_palette))

# 分箱聚合 (100个 Progress 点)
time_complex['Progress_Bin'] = (time_complex['Progress'] * 100).astype(int)

# 聚合统计：0.25 0.5 0.75三个核心分位数
agg_metrics = time_complex.groupby(['ELO_Group', 'Progress_Bin'])['Cognitive_Speed'].agg(
    mean='mean',
    q25=lambda x: x.quantile(0.25),
    q50='median',
    q75=lambda x: x.quantile(0.75)
).reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 18), sharex=True)

# 均值对比图
for group in elo_order:
    g_data = agg_metrics[agg_metrics['ELO_Group'] == group]
    ax1.plot(g_data['Progress_Bin'], g_data['mean'], 
             color=color_map[group], linewidth=2.5, label=group)

ax1.set_title(r'Mean Cognitive Speed over Game Progress ($\Delta/t$)', fontsize=16, fontweight='bold')
ax1.set_ylabel('Mean Speed (cp/sec)', fontsize=13)
ax1.set_ylim(0, 5000)  # 均值天花板设为 5000
ax1.grid(True, linestyle=':', alpha=0.4)
ax1.legend(title='ELO Group', bbox_to_anchor=(1.01, 1), loc='upper left')

# 分位数分布图
for group in elo_order:
    g_data = agg_metrics[agg_metrics['ELO_Group'] == group]
    color = color_map[group]
    
    # q50 实线
    ax2.plot(g_data['Progress_Bin'], g_data['q50'], color=color, 
             linestyle='-', linewidth=3, alpha=0.9)
    
    # q25 & q75 虚线
    ax2.plot(g_data['Progress_Bin'], g_data['q25'], color=color, 
             linestyle='--', linewidth=1.0, alpha=0.9)
    ax2.plot(g_data['Progress_Bin'], g_data['q75'], color=color, 
             linestyle='--', linewidth=1.0, alpha=0.9)

ax2.set_title(r'Quantile Dynamics (Solid: Median ; Dash: 0.25，0.75 Percentiles)', fontsize=16, fontweight='bold')
ax2.set_ylabel('Cognitive Speed (cp/sec)', fontsize=13)
ax2.set_xlabel('Game Progress (%)', fontsize=13)
ax2.set_ylim(0, 300)  # 分位数天花板设为 1200，观察中位区间的细微差别
ax2.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

棋局复杂度与棋手失误类型的关系

# Complexity 高低 → Blunder / Mistake 发生概率

In [ ]:
pd.set_option('display.max_columns', None)
complex_drop_force.head()

In [ ]:
df_moves.columns

In [ ]:
def plot_error_rate_by_progress_fast(df, bins=100):    
    # 分箱计算
    bin_indices = np.clip((df['Progress'].values * bins).astype(int), 0, bins - 1)
    
    # 避免超出内存，仅提取要用的数据
    temp_df = pd.DataFrame({'ELO_Group': df['ELO_Group'].values, 'Bin_Idx': bin_indices, 'Is_Error': df['Is_Error'].values})
    
    # 各个 ELO 组的统计量
    group_stats = temp_df.groupby(['ELO_Group', 'Bin_Idx'], observed=False)['Is_Error'].agg(['sum', 'count']).reset_index()
    group_stats['Error_Rate'] = group_stats['sum'] / group_stats['count']
    group_stats['Progress_Mid'] = (group_stats['Bin_Idx'] + 0.5) / bins

    # 全局 (Global) 的统计量， 用已聚合的 group_stats 二次聚合
    global_stats = group_stats.groupby('Bin_Idx')[['sum', 'count']].sum().reset_index()
    global_stats['Error_Rate'] = global_stats['sum'] / global_stats['count']
    global_stats['Progress_Mid'] = (global_stats['Bin_Idx'] + 0.5) / bins

    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    
    # 画全局均线 (加粗、黑色、虚线，放在底层或顶层都可以，这里设为较高的 zorder 让它明显一点)
    ax.plot(global_stats['Progress_Mid'], global_stats['Error_Rate'], 
        color='black', linestyle='--', linewidth=3, alpha=0.7, label='Global Average',zorder=3)
    # ELO 分组线
    for i, group in enumerate(elo_order):
        group_data = group_stats[group_stats['ELO_Group'] == group]
        group_data = group_data.sort_values('Progress_Mid')
        
        ax.plot(group_data['Progress_Mid'],group_data['Error_Rate'], 
            color=distinct_palette[i], linewidth=1.5, alpha=0.85, label=group,zorder=2)
    
    plt.title("Error Rate Across Game Progress", fontsize=14, fontweight='bold')
    plt.xlabel("Game Progress (0.0=Start, 1.0=End)", fontsize=12)
    plt.ylabel("P(Error)", fontsize=12)
    plt.xlim(0, 1)
    max_y = max(group_stats['Error_Rate'].max(), global_stats['Error_Rate'].max())
    plt.ylim(0, max_y * 1.15) 
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9)
    text_str = "P(Error) = Count(Mistake/Blunder) / Count(Total)"
    props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8)
    ax.text(0.98, 0.03, text_str, transform=ax.transAxes, fontsize=11,verticalalignment='bottom', horizontalalignment='right', bbox=props, zorder=4)

    plt.tight_layout()
    plt.show()

plot_error_rate_by_progress_fast(complex_drop_force, bins=200)

In [ ]:
total_moves = len(complex_drop_force)
total_errors = complex_drop_force['Is_Error'].sum()
overall_error_rate = total_errors / total_moves

print(f"总步数: {total_moves}")
print(f"总错误步数: {total_errors}")
print(f"整体错误率: {overall_error_rate:.2%}")

In [ ]:
# 这张图是为了看mistake blunder主要分布在游戏的哪个阶段比较多，纵坐标的百分比占比
# 可以看出开局阶段很少有失误， 但是中后局选手出错概率增加很多
def plot_hist_with_values(df, col, bins=50, kde=False, title=None, xlabel=None, ylabel='Relative Frequency (Probability)', color='blue'):
    """绘制直方图/柱状图并打印每个区间的具体步数。"""
    values = df[col].dropna()
    
    counts, bin_edges = np.histogram(values, bins=bins)
    
    print(f"=== {title if title else col} 每个区间步数 ===")
    for i in range(0, len(counts), 2):
        left = f"{bin_edges[i]:.3f}-{bin_edges[i+1]:.3f}: {counts[i]}"
        if i + 1 < len(counts):
            right = f"{bin_edges[i+1]:.3f}-{bin_edges[i+2]:.3f}: {counts[i+1]}"
            print(f"{left:<30} | {right}")
        else:
            print(left)

    plt.figure(figsize=(8,5))
    sns.histplot(values, bins=bins, kde=kde, color=color, stat='probability')
    plt.title(title if title else col)
    plt.xlabel(xlabel if xlabel else col)
    plt.ylabel(ylabel)
    plt.show() 
# mistake blunder distribution
# 只保留错误步
errors_df = complex_drop_force[complex_drop_force['Is_Error']==1]

plot_hist_with_values(df=errors_df, col='Progress', bins=50, kde=True, title="Mistake / Blunder 's Game-Progress distribution", xlabel='Relative Progress', ylabel='Proportion',color='orange')

In [ ]:
τ_map = {
    '>=3000': 65.8,
    '2800-3000': 85.4,
    '2600-2800': 113.2,
    '2400-2600': 129.0,
    '2200-2400': 144.7,
    '2000-2200': 185.6,
    '<2000': 170.8,     # 手动补全
    'Unknown': 120.0    # 默认值
}
τ_map = {
    '>=3000': 105.3,
    '2800-3000': 113.3,
    '2600-2800': 118.5,
    '2400-2600': 122.5,
    '2200-2400': 138.3,
    '2000-2200': 129.8,
    '<2000': 170.8,     # 手动补全
    'Unknown': 120.0    # 默认值
}
elo_params = {
    '>=3000':    {'A': 0.772, 'tau': 105.3},
    '2800-3000': {'A': 0.749, 'tau': 113.3},
    '2600-2800': {'A': 0.733, 'tau': 118.5},
    '2400-2600': {'A': 0.712, 'tau': 122.5},
    '2200-2400': {'A': 0.691, 'tau': 138.3},
    '2000-2200': {'A': 0.679, 'tau': 129.8},
    '<2000':     {'A': 0.656, 'tau': 170.8}
}

def plot_error_rate_with_tau_markers(df, delta_col='Δ', max_delta=500, bins=100): 
    # 极速分箱逻辑保持不变
    vals = np.clip(df[delta_col].values, 0, max_delta)
    bin_indices = np.clip((vals / max_delta * bins).astype(int), 0, bins - 1)
    
    temp_df = pd.DataFrame({
        'ELO_Group': df['ELO_Group'].values, 
        'Bin_Idx': bin_indices, 
        'Is_Error': df['Is_Error'].values
    })
    
    group_stats = temp_df.groupby(['ELO_Group', 'Bin_Idx'], observed=False)['Is_Error'].agg(['sum', 'count']).reset_index()
    group_stats['Error_Rate'] = group_stats['sum'] / group_stats['count']
    bin_width = max_delta / bins
    group_stats['Delta_Mid'] = (group_stats['Bin_Idx'] + 0.5) * bin_width

    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    
    # 画 ELO 分组线，并在 tau 处打点
    for i, group in enumerate(elo_order):
        group_data = group_stats[group_stats['ELO_Group'] == group]
        group_data = group_data.sort_values('Delta_Mid')
        color = distinct_palette[i]
        tau_val = τ_map[group]
        
        # 画主线
        ax.plot(group_data['Delta_Mid'], group_data['Error_Rate'], 
                color=color, linewidth=1.8, alpha=0.8, label=f"{group}")
        
        # 寻找距离 tau 最近的 x 坐标点，为了画那个点
        closest_idx = (np.abs(group_data['Delta_Mid'] - tau_val)).argmin()
        closest_row = group_data.iloc[closest_idx]
        
        # 在线上画一个显眼的实心圆点代表 tau
        ax.plot(closest_row['Delta_Mid'], closest_row['Error_Rate'], 
                marker='o', markersize=8, color=color, markeredgecolor='white', markeredgewidth=1.5, zorder=5)
        
        # 画一根淡淡的垂直虚线指向 X 轴，强调 tau 的位置
        ax.vlines(x=tau_val, ymin=0, ymax=closest_row['Error_Rate'], 
                  color=color, linestyle=':', alpha=0.5, zorder=1)

    plt.title(f"Error Rate Across {delta_col} (Dots represent Group-specific $\\tau$ thresholds)", fontsize=14, fontweight='bold')
    plt.xlabel(f"{delta_col} (cp)", fontsize=12)
    plt.ylabel("P(Error)", fontsize=12)
    
    plt.xlim(0, max_delta)
    # 只看 500cp 以内，所以 Y 轴最高基本在 0.25 左右，动态截取让图更好看
    max_y = group_stats['Error_Rate'].max()
    plt.ylim(0, max_y * 1.1) 
    
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9, title='ELO Group')
    
    text_str = "Circular markers indicate the calibrated $\\tau$ value for each group."
    props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8)
    ax.text(0.98, 0.90, text_str, transform=ax.transAxes, fontsize=11,
            verticalalignment='bottom', horizontalalignment='right', bbox=props, zorder=4)

    plt.tight_layout()
    plt.show()

# 截断在 500cp，看最核心的决策区
# plot_error_rate_with_tau_markers(complex_drop_force, delta_col='Δ', max_delta=1000, bins=80)

def plot_error_rate_with_tau_markers_normalized(df, delta_col='Δ', max_delta=500, bins=100): 
    # 极速分箱逻辑保持不变
    vals = np.clip(df[delta_col].values, 0, max_delta)
    bin_indices = np.clip((vals / max_delta * bins).astype(int), 0, bins - 1)
    
    temp_df = pd.DataFrame({
        'ELO_Group': df['ELO_Group'].values, 
        'Bin_Idx': bin_indices, 
        'Is_Error': df['Is_Error'].values
    })
    
    group_stats = temp_df.groupby(['ELO_Group', 'Bin_Idx'], observed=False)['Is_Error'].agg(['sum', 'count']).reset_index()
    group_stats['Error_Rate'] = group_stats['sum'] / group_stats['count']
    bin_width = max_delta / bins
    group_stats['Delta_Mid'] = (group_stats['Bin_Idx'] + 0.5) * bin_width

    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    
    print(f"{'ELO Group':<12} | {'Marker X (≈τ)':<15} | {'Marker Y (P/A)':<15}")
    print("-" * 50)

    # 画 ELO 分组线，并在 tau 处打点
    max_y_plotted = 0 # 用于动态调整Y轴上限
    
    for i, group in enumerate(elo_order):
        # 使用 copy 防止修改原始 warning
        group_data = group_stats[group_stats['ELO_Group'] == group].copy()
        group_data = group_data.sort_values('Delta_Mid')
        color = distinct_palette[i]
        
        tau_val = elo_params[group]['tau']
        A_val = elo_params[group]['A']
        
        # 【核心修改点】：纵坐标除以 A 进行归一化
        group_data['Normalized_Error'] = group_data['Error_Rate'] / A_val
        
        # 记录用于设置 Y 轴上限的最大值
        current_max = group_data['Normalized_Error'].max()
        if current_max > max_y_plotted:
            max_y_plotted = current_max

        # 画主线 (纵坐标已经是归一化后的了)
        ax.plot(group_data['Delta_Mid'], group_data['Normalized_Error'], 
                color=color, linewidth=1.8, alpha=0.8, label=f"{group}")
        
        # 寻找距离 tau 最近的 x 坐标点，为了画那个点
        closest_idx = (np.abs(group_data['Delta_Mid'] - tau_val)).argmin()
        closest_row = group_data.iloc[closest_idx]
        
        marker_x = closest_row['Delta_Mid']
        marker_y = closest_row['Normalized_Error']
        
        # 【核心修改点】：打印标点的坐标
        print(f"{group:<12} | {marker_x:<15.2f} | {marker_y:<15.4f}")
        
        # 在线上画一个显眼的实心圆点代表 tau
        ax.plot(marker_x, marker_y, 
                marker='o', markersize=8, color=color, markeredgecolor='white', markeredgewidth=1.5, zorder=5)
        
        # 画一根淡淡的垂直虚线指向 X 轴，强调 tau 的位置
        ax.vlines(x=tau_val, ymin=0, ymax=marker_y, 
                  color=color, linestyle=':', alpha=0.5, zorder=1)

    plt.title(f"Capacity-Normalized Error Rate Across {delta_col}", fontsize=14, fontweight='bold')
    plt.xlabel(f"{delta_col} (cp)", fontsize=12)
    plt.ylabel("Normalized P(Error) [ P / A ]", fontsize=12) # 更新了纵坐标标签
    
    plt.xlim(0, max_delta)
    # 动态调整 Y 轴上限，留出 10% 的余量
    plt.ylim(0, max_y_plotted * 1.1) 
    
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9, title='ELO Group')
    
    text_str = (
        "Y-axis is normalized by group-specific capacity (A).\n"
        "Circular markers indicate the calibrated $\\tau$ value."
    )
    props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8)
    ax.text(0.98, 0.90, text_str, transform=ax.transAxes, fontsize=11,
            verticalalignment='bottom', horizontalalignment='right', bbox=props, zorder=4)

    plt.tight_layout()
    plt.show()

# 跑起来看看效果和打印的坐标值！
plot_error_rate_with_tau_markers_normalized(complex_drop_force, delta_col='Δ', max_delta=500, bins=100)

和τ重新扯上关系，横坐标除以τ看分布是否会趋向一致    
感觉效果不好，而且后尾部噪声多    
也许后面会试试log函数？

In [ ]:
τ_map = {
    '>=3000': 105.3,
    '2800-3000': 113.3,
    '2600-2800': 118.5,
    '2400-2600': 122.5,
    '2200-2400': 138.3,
    '2000-2200': 129.8,
    '<2000': 170.8,     # 手动补全
    'Unknown': 120.0    # 默认值
}
def plot_error_rate_by_normalized_delta_fast(df, delta_col='Δ', max_multiplier=8, bins=100): 
    """横轴为 Δ/τ，纵轴为 P(Error) 的折线图"""
    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    
    # 用于累加计算 Global Average
    global_counts = np.zeros(bins)
    global_errors = np.zeros(bins)
    
    # 提前算出横轴的 X 坐标 (每个 bin 的中心点)
    bin_width = max_multiplier / bins
    x_mids = (np.arange(bins) + 0.5) * bin_width

    # 将常用的列提取为 numpy array，极大加速循环内的切片速度
    all_elos = df['ELO_Group'].values
    all_deltas = df[delta_col].values
    all_errors = df['Is_Error'].values

    # 按 ELO 组循环处理
    for i, group in enumerate(elo_order):
        tau = τ_map[group]
        
        # 1. 提取当前组数据
        mask = all_elos == group
        group_deltas = all_deltas[mask]
        group_is_error = all_errors[mask]
        
        # 2. 核心数学转换：归一化 (无量纲化)
        # 这就是把不同水平选手的曲线拉平的关键！
        norm_deltas = group_deltas / tau
        
        # 3. 极速分箱映射 (将 0 ~ max_multiplier 映射到 0 ~ bins-1)
        vals = np.clip(norm_deltas, 0, max_multiplier)
        bin_indices = np.clip((vals / max_multiplier * bins).astype(int), 0, bins - 1)
        
        # 4. 使用 numpy 底层极速聚合 (完全替代 groupby)
        # counts: 这个箱子里有多少步
        counts = np.bincount(bin_indices, minlength=bins)
        # errors: 这个箱子里有多少错误步
        errors = np.bincount(bin_indices, weights=group_is_error, minlength=bins)
        
        # 累加到全局数据中
        global_counts += counts
        global_errors += errors
        
        # 5. 计算错误率 (忽略除以 0 的警告，后续画图会过滤掉)
        with np.errstate(invalid='ignore'):
            error_rates = errors / counts
            
        # 过滤掉 count == 0 的无效点，防止画图时出现断点或掉到 0
        valid = counts > 0
        
        # 画出当前组别的线，并在图例中标注对应的 tau
        ax.plot(x_mids[valid], error_rates[valid], 
                color=distinct_palette[i], linewidth=1.5, alpha=0.85, 
                label=f"{group} ($\\tau$={tau})", zorder=2)

    # 6. 计算并绘制 Global Average
    with np.errstate(invalid='ignore'):
        global_error_rates = global_errors / global_counts
    valid_global = global_counts > 0
    ax.plot(x_mids[valid_global], global_error_rates[valid_global], 
            color='black', linestyle='--', linewidth=3, alpha=0.7, 
            label='Global Average', zorder=3)

    # 7. 设置图表轴与标签
    plt.title("Normalized Error Rate: Evidence of $\\tau$ as a Characteristic Scale", fontsize=15, fontweight='bold')
    plt.xlabel(f"Normalized Decision Margin ( $\\Delta$ / $\\tau$ )", fontsize=13)
    plt.ylabel("P(Error)", fontsize=13)
    
    plt.xlim(0, max_multiplier)
    # 动态调整 Y 轴上限
    max_y = np.nanmax(global_error_rates)
    plt.ylim(0, max_y * 1.5) 
    
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9, title="ELO Group & Threshold")
    
    # 说明框
    text_str = (
        "X-axis is dimensionless.\n"
        "If the curves collapse together,\n"
        "it validates $\\tau$ as the universal scaling\n"
        "factor for blunder probability."
    )
    props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8)
    ax.text(0.98, 0.80, text_str, transform=ax.transAxes, fontsize=11,
            verticalalignment='bottom', horizontalalignment='right', bbox=props, zorder=4)

    plt.tight_layout()
    plt.show()

plot_error_rate_by_normalized_delta_fast(complex_drop_force, delta_col='Δ', max_multiplier=8, bins=100)

横坐标非线性处理， 同时纵坐标归一化处理 但感觉也就那样

In [ ]:
elo_params = {
    '>=3000':    {'A': 0.772, 'tau': 105.3}, # A值需要你填入该组错误率大致的最高点
    '2800-3000': {'A': 0.749, 'tau': 113.3},
    '2600-2800': {'A': 0.733, 'tau': 118.5},
    '2400-2600': {'A': 0.712, 'tau': 122.5},
    '2200-2400': {'A': 0.691, 'tau': 138.3},
    '2000-2200': {'A': 0.679, 'tau': 129.8},
    '<2000':     {'A': 0.656, 'tau': 170.8}
}

def plot_ultimate_collapse(df, delta_col='Δ', max_delta=800, bins=80): 
    plt.figure(figsize=(12, 7))
    ax = plt.gca()

    for i, group in enumerate(elo_order):
        tau = elo_params[group]['tau']
        A_ceiling = elo_params[group]['A']
        
        # 提取数据
        group_df = df[(df['ELO_Group'] == group) & (df[delta_col] <= max_delta)]
        if len(group_df) == 0: continue
            
        deltas = group_df[delta_col].values
        errors = group_df['Is_Error'].values
        
        # 1. 温和的非线性 X 轴压缩: 1 - exp(-Δ/τ)
        # 这样所有的横坐标都会被安全地限制在 0 到 1 之间
        x_mapped = 1 - np.exp(-deltas / tau)
        
        # 将 0-1 的区间分成 bins 份
        bin_indices = np.clip((x_mapped * bins).astype(int), 0, bins - 1)
        
        counts = np.bincount(bin_indices, minlength=bins)
        err_counts = np.bincount(bin_indices, weights=errors, minlength=bins)
        
        with np.errstate(invalid='ignore'):
            raw_error_rate = err_counts / counts
            # 2. 你的神来之笔：纵坐标除以 A 
            normalized_error_rate = raw_error_rate / A_ceiling
            
        valid = counts > 20 # 过滤掉样本太少的噪声点
        x_mids = (np.arange(bins) + 0.5) / bins
        
        # 画线
        ax.plot(x_mids[valid], normalized_error_rate[valid], 
                color=distinct_palette[i], linewidth=2, alpha=0.7, label=group)

    plt.title("Ultimate Collapse: Non-linear Standardized X and Capacity-Normalized Y", fontsize=14, fontweight='bold')
    plt.xlabel(r"Standardized Difficulty Threshold: $1 - \exp(-\Delta / \tau)$", fontsize=12)
    plt.ylabel(r"Relative Error Saturation: $P(Error) / A$", fontsize=12)
    
    plt.xlim(0, 1)
    plt.ylim(0, 1.2) # 饱和度通常在 1.0 左右波动
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', title="ELO Group")

    plt.tight_layout()
    plt.show()

plot_ultimate_collapse(complex_drop_force, delta_col='Δ', max_delta=800, bins=80)

In [ ]:
df_moves.head()

In [ ]:
def get_relative_stage(progress):
    if progress <= 0.33:
        return 'Opening (0-33%)'
    elif progress <= 0.66:
        return 'Middlegame (33-66%)'
    else:
        return 'Endgame (66-100%)'
df_moves['Relative_Stage(temp)'] = df_moves['Progress'].apply(get_relative_stage)

In [ ]:
plt.figure(figsize=(10, 6))

# 画 Group-specific S
sns.lineplot(
    x=df_moves['Progress'].round(2),
    y=df_moves['S'],
    estimator='mean',
    errorbar=None,
    label='S (Group)'
)

# 自动找所有 Global S 列
global_s_cols = [col for col in df_moves.columns if col.startswith('S(Global=')]

# 画 Global S
for col in global_s_cols:
    sns.lineplot(
        x=df_moves['Progress'].round(2),
        y=df_moves[col],
        estimator='mean',
        errorbar=None,
        label=col.replace('S(', '').replace(')', '')
    )

plt.ylim(0, 1)
plt.title("Complexity (S) Trend over Game Progress")
plt.xlabel("Game Progress (0.0 = Start, 1.0 = Finish)")
plt.ylabel("Average Complexity (S)")
plt.axvline(0.33, color='r', linestyle='--', alpha=0.3, label='1/3 Split')
plt.axvline(0.66, color='r', linestyle='--', alpha=0.3)
plt.legend(fontsize=6)
plt.grid(True, alpha=0.3)
plt.show()


横坐标换成原始Δ再看一下

In [ ]:
elo_order = ['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']
distinct_palette = sns.color_palette("Set1", len(elo_order))

def plot_raw_delta_over_progress_fast(df, delta_col='Δ', bins=100, clip_delta=2000):
    """绘制不同对局进度下的 平均原始 Δ 趋势"""
    # 极速分箱
    bin_indices = np.clip((df['Progress'].values * bins).astype(int), 0, bins - 1)
    
    # 截断极端的将死分数
    clipped_deltas = np.clip(df[delta_col].values, 0, clip_delta)
    temp_df = pd.DataFrame({
        'ELO_Group': df['ELO_Group'].values,
        'Bin_Idx': bin_indices,
        'Delta': clipped_deltas
    })
    
    # 分组计算均值
    group_stats = temp_df.groupby(['ELO_Group', 'Bin_Idx'], observed=False)['Delta'].mean().reset_index()
    group_stats['Progress_Mid'] = (group_stats['Bin_Idx'] + 0.5) / bins
    global_stats = temp_df.groupby('Bin_Idx')['Delta'].mean().reset_index()
    global_stats['Progress_Mid'] = (global_stats['Bin_Idx'] + 0.5) / bins

    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    ax.plot(global_stats['Progress_Mid'], global_stats['Delta'], 
            color='black', linestyle='--', linewidth=3, alpha=0.7, label='Global Average', zorder=3)
    for i, group in enumerate(elo_order):
        group_data = group_stats[group_stats['ELO_Group'] == group]
        group_data = group_data.sort_values('Progress_Mid')
        
        ax.plot(group_data['Progress_Mid'], group_data['Delta'], 
                color=distinct_palette[i], linewidth=1.5, alpha=0.85, label=group, zorder=2)
    
    plt.title(f"Average {delta_col} Trend over Game Progress", fontsize=14, fontweight='bold')
    plt.xlabel("Game Progress (0.0 = Start, 1.0 = Finish)", fontsize=12)
    plt.ylabel(f"Average {delta_col} (cp)", fontsize=12)
    plt.xlim(0, 1)
    plt.ylim(0, max(group_stats['Delta'].max(), global_stats['Delta'].max()) * 1.15) 
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9, title="ELO Group")
    plt.tight_layout()
    plt.show()

plot_raw_delta_over_progress_fast(complex_drop_force, delta_col='Δ', bins=100, clip_delta=2000)

In [ ]:
plt.figure(figsize=(10, 6))

# 画 Group-specific S
sns.lineplot(
    x=complex_drop_force['Progress'].round(2),
    y=complex_drop_force['S'],
    estimator='mean',
    errorbar=None,
    label='S (Group)'
)

# 自动找所有 Global S 列
global_s_cols = [col for col in complex_drop_force.columns if col.startswith('S(Global=')]

# 画 Global S
for col in global_s_cols:
    sns.lineplot(
        x=complex_drop_force['Progress'].round(2),
        y=complex_drop_force[col],
        estimator='mean',
        errorbar=None,
        label=col.replace('S(', '').replace(')', '')
    )

plt.ylim(0, 1)
plt.title("Complexity (S) Trend over Game Progress")
plt.xlabel("Game Progress (0.0 = Start, 1.0 = Finish)")
plt.ylabel("Average Complexity (S)")
plt.axvline(0.33, color='r', linestyle='--', alpha=0.3, label='1/3 Split')
plt.axvline(0.66, color='r', linestyle='--', alpha=0.3)
plt.legend(fontsize=6)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
n_bins = 50
df_moves['Progress_bin'] = pd.cut(df_moves['Progress'], bins=n_bins, labels=False)

# --- 绘图准备 ---
n_cols = 3
plot_cols = ['S'] + global_s_cols

n_plots = len(plot_cols)
n_rows = math.ceil((n_plots + 1) / n_cols) # 确保为图例留出位置

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()
palette = sns.color_palette("tab10", n_colors=len(elo_order))

for i, col in enumerate(plot_cols):
    ax = axes[i]
    
    # 整体平均线
    overall_avg = df_moves.groupby('Progress_bin')[col].mean()
    ax.plot(overall_avg.index / n_bins, overall_avg.values, color='black', linewidth=1, label='Overall', zorder=10)

    # 各 ELO 组曲线
    for j, elo in enumerate(elo_order):
        sub_df = df_moves[df_moves['ELO_Group'] == elo]
        group_avg = sub_df.groupby('Progress_bin')[col].mean()
        ax.plot(group_avg.index / n_bins, group_avg.values, color=palette[j], label=str(elo), alpha=0.8)

    # 样式美化
    ax.set_title("Proposed: Group-specific S" if col=='S' else f"Global Δ₀ = {col.split('=')[1].replace(')','')}", fontsize=12)
    ax.set_xlabel("Game Progress (0=Start, 1=Finish)")
    ax.set_ylabel("Average Complexity (S)")
    ax.set_ylim(0, 1)
    ax.axvline(0.33, color='r', linestyle='--', alpha=0.3)
    ax.axvline(0.66, color='r', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3)

# 图例
legend_ax = axes[n_plots] 
legend_ax.axis('off') 
handles, labels = axes[0].get_legend_handles_labels()
legend_ax.legend(handles, labels, loc='center', title="ELO Groups", 
                  fontsize=10, title_fontsize=11, frameon=True, shadow=True)

# --- 清理多余的空白子图 ---
for j in range(n_plots + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
plt.show()

In [ ]:
df_moves.to_csv('peak.csv')

In [ ]:
complex_drop_force.to_csv('dropforce.csv')